# NB_lab_test_trends — Gold: Lab Test Trends

Builds `adb_dev_healthcare.gold.lab_test_trends` — **one row per test type per calendar month** — aggregating lab results with month-over-month abnormal rate trends via LAG.

**Load Type:** Full rebuild (CREATE OR REPLACE TABLE AS SELECT)

In [0]:
%sql
CREATE OR REPLACE TABLE adb_dev_healthcare.gold.lab_test_trends AS

WITH base_agg AS (
  SELECT
    lr.test_name,
    YEAR(lr.test_date)                                                  AS year,
    MONTH(lr.test_date)                                                 AS month,
    COUNT(*)                                                            AS total_tests,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)          AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)          AS critical_test_count
  FROM adb_dev_healthcare.silver.lab_results lr
  GROUP BY lr.test_name, YEAR(lr.test_date), MONTH(lr.test_date)
),

enriched AS (
  SELECT
    ba.test_name,
    ba.year,
    ba.month,
    c.month_name,
    CONCAT('Q', c.quarter)                                              AS quarter_name,
    ba.total_tests,
    ba.abnormal_test_count,
    ba.critical_test_count,
    ROUND(ba.abnormal_test_count / ba.total_tests * 100, 2)             AS abnormal_rate_pct,
    ROUND(ba.critical_test_count / ba.total_tests * 100, 2)             AS critical_rate_pct
  FROM      base_agg                                     ba
  LEFT JOIN adb_dev_healthcare.silver.calendar             c
         ON c.date_key = CAST(DATE_FORMAT(DATE_TRUNC('MONTH', MAKE_DATE(ba.year, ba.month, 1)), 'yyyyMMdd') AS INT)
)

SELECT
  test_name,
  year,
  month,
  month_name,
  quarter_name,
  total_tests,
  abnormal_test_count,
  critical_test_count,
  abnormal_rate_pct,
  critical_rate_pct,
  LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month)  AS prev_month_abnormal_rate_pct,
  ROUND(
    abnormal_rate_pct
    - LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month),
    2
  )                                                                          AS abnormal_rate_change_pct
FROM enriched;

In [0]:
updated_df = spark.table("adb_dev_healthcare.gold.lab_test_trends")
records_count = updated_df.count()
dbutils.notebook.exit(str(records_count))